# Fine-Tuning Object Detectors

Pretrained detectors work well on the COCO categories they were trained on, but real projects
almost always require custom classes. This notebook walks through annotation formats,
dataset preparation, augmentation, and fine-tuning a YOLOv8 detector on a custom dataset.

In [ ]:
# !pip install ultralytics albumentations pillow
import json
import os
import random
import shutil
from pathlib import Path

import albumentations as A
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

## Annotation Formats

Three formats dominate detection annotation. Knowing all three is practical because
datasets arrive in different formats and tools expect different ones.

### COCO JSON
A single JSON file for the whole dataset split. Annotations are stored as a flat list
referencing images by ID. Bounding boxes are `[x_min, y_min, width, height]` in absolute pixels.

```json
{
  "images": [{"id": 1, "file_name": "img001.jpg", "width": 640, "height": 480}],
  "categories": [{"id": 1, "name": "dog"}, {"id": 2, "name": "cat"}],
  "annotations": [
    {"id": 1, "image_id": 1, "category_id": 1, "bbox": [120, 80, 200, 150]}
  ]
}
```

### YOLO TXT
One `.txt` file per image, same filename. Each line is one object:
`class_id cx cy w h` where all coordinates are normalized to [0, 1].

```
0 0.34 0.45 0.31 0.42
1 0.71 0.22 0.15 0.18
```

### Pascal VOC XML
One XML file per image. Coordinates are absolute pixel values, stored as `xmin/ymin/xmax/ymax`.

```xml
<annotation>
  <filename>img001.jpg</filename>
  <size><width>640</width><height>480</height><depth>3</depth></size>
  <object>
    <name>dog</name>
    <bndbox><xmin>120</xmin><ymin>80</ymin><xmax>320</xmax><ymax>230</ymax></bndbox>
  </object>
</annotation>
```

## Dataset Structure: YOLO Folder Layout and train.yaml

The ultralytics trainer expects a specific folder layout. Images and label files must share the same filename stem. A `dataset.yaml` file tells the trainer where the data lives and defines the class names.

In [ ]:
import os, textwrap
from pathlib import Path

# Required folder structure:
#
#   custom_dataset/
#     images/
#       train/    <- .jpg / .png files
#       val/
#     labels/
#       train/    <- one .txt per image (same stem as the image)
#       val/
#     dataset.yaml

DATASET_STRUCTURE = """
custom_dataset/
  images/
    train/
      cat001.jpg
      dog001.jpg
      ...
    val/
      cat100.jpg
      ...
  labels/
    train/
      cat001.txt     <- YOLO label: one line per object
      dog001.txt
      ...
    val/
      cat100.txt
      ...
  dataset.yaml
"""

print('Required YOLO dataset folder structure:')
print(DATASET_STRUCTURE)

# Each label file contains one line per object:
#   class_id  cx  cy  width  height
# All coordinates are normalized to [0, 1] relative to the image dimensions.
EXAMPLE_LABEL = """
# cat001.txt (image is 640x480, cat box is pixel x=100..300, y=80..220)
0 0.3125 0.3125 0.3125 0.2917

# Decoded:
#   class_id = 0  (first class in your YAML)
#   cx = (100+300)/2 / 640 = 200/640 = 0.3125
#   cy = (80+220)/2  / 480 = 150/480 = 0.3125
#   w  = (300-100)   / 640 = 200/640 = 0.3125
#   h  = (220-80)    / 480 = 140/480 = 0.2917
"""
print('Example label file content:')
print(EXAMPLE_LABEL)

# Example dataset.yaml
EXAMPLE_YAML = """
path: /path/to/custom_dataset   # absolute path to dataset root
train: images/train
val:   images/val

nc: 2                            # number of classes
names: ['cat', 'dog']           # class names in order (index 0 = cat, 1 = dog)
"""
print('Example dataset.yaml:')
print(EXAMPLE_YAML)

print('Rules:')
print('  - Every image in images/train/ must have a matching .txt in labels/train/')
print('  - Images with no objects get an empty .txt file (not missing, empty)')
print('  - Class IDs are 0-indexed and must match the "names" list in the YAML')

## Conversion Helper: VOC XML to YOLO Format

Pascal VOC XML uses absolute pixel coordinates in `xmin/ymin/xmax/ymax` format. The function below reads all `.xml` files from a VOC annotation directory and writes corresponding `.txt` YOLO label files.

In [ ]:
import xml.etree.ElementTree as ET
import json

def convert_voc_to_yolo(voc_xml_dir, output_label_dir, class_names):
    """
    Convert Pascal VOC XML annotation files to YOLO TXT format.

    Args:
        voc_xml_dir:    directory containing .xml annotation files
        output_label_dir: directory to write .txt YOLO label files
        class_names:    list of class names in order, e.g. ['cat', 'dog']
                        Must include every class that appears in the XMLs.

    The function writes one .txt file per .xml file. Bounding box coordinates
    are normalized to [0, 1].  Classes not in class_names are skipped.
    """
    os.makedirs(output_label_dir, exist_ok=True)
    xml_files = list(Path(voc_xml_dir).glob('*.xml'))
    class_to_id = {name: i for i, name in enumerate(class_names)}

    converted = skipped = 0

    for xml_path in xml_files:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Image dimensions from <size>
        size_node = root.find('size')
        img_w = int(size_node.find('width').text)
        img_h = int(size_node.find('height').text)

        lines = []
        for obj in root.findall('object'):
            cls_name = obj.find('name').text.strip()
            if cls_name not in class_to_id:
                skipped += 1
                continue

            cls_id = class_to_id[cls_name]
            bbox   = obj.find('bndbox')
            xmin   = float(bbox.find('xmin').text)
            ymin   = float(bbox.find('ymin').text)
            xmax   = float(bbox.find('xmax').text)
            ymax   = float(bbox.find('ymax').text)

            # Convert to YOLO normalized format
            cx = (xmin + xmax) / 2.0 / img_w
            cy = (ymin + ymax) / 2.0 / img_h
            w  = (xmax - xmin) / img_w
            h  = (ymax - ymin) / img_h

            lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        stem = xml_path.stem
        out_path = Path(output_label_dir) / f'{stem}.txt'
        with open(out_path, 'w') as f:
            f.write('\n'.join(lines))
        converted += 1

    print(f'convert_voc_to_yolo: converted {converted} files, skipped {skipped} unknown classes')
    print(f'Output directory: {output_label_dir}')


# --- Demo: write a synthetic VOC XML and convert it ---
DEMO_XML = """\
<annotation>
  <filename>sample.jpg</filename>
  <size><width>640</width><height>480</height><depth>3</depth></size>
  <object>
    <name>cat</name>
    <bndbox><xmin>100</xmin><ymin>80</ymin><xmax>300</xmax><ymax>230</ymax></bndbox>
  </object>
  <object>
    <name>dog</name>
    <bndbox><xmin>350</xmin><ymin>150</ymin><xmax>500</xmax><ymax>350</ymax></bndbox>
  </object>
</annotation>
"""

os.makedirs('/tmp/voc_demo', exist_ok=True)
with open('/tmp/voc_demo/sample.xml', 'w') as f:
    f.write(DEMO_XML)

convert_voc_to_yolo(
    voc_xml_dir='/tmp/voc_demo',
    output_label_dir='/tmp/voc_yolo_out',
    class_names=['cat', 'dog'],
)

with open('/tmp/voc_yolo_out/sample.txt') as f:
    print('\nConverted YOLO label (sample.txt):')
    print(f.read())

In [ ]:
def coco_bbox_to_yolo(bbox, img_width, img_height):
    """Convert COCO [x_min, y_min, w, h] to YOLO [cx, cy, w, h] normalized."""
    x_min, y_min, w, h = bbox
    cx = (x_min + w / 2) / img_width
    cy = (y_min + h / 2) / img_height
    nw = w / img_width
    nh = h / img_height
    return cx, cy, nw, nh


def convert_coco_to_yolo(coco_json_path, output_label_dir, category_mapping=None):
    """
    Convert a COCO annotation file to per-image YOLO txt files.
    category_mapping: optional dict {coco_id: yolo_class_id}.
    If None, COCO category IDs are used directly (0-indexed).
    """
    os.makedirs(output_label_dir, exist_ok=True)

    with open(coco_json_path) as f:
        coco = json.load(f)

    id_to_image = {img["id"]: img for img in coco["images"]}

    if category_mapping is None:
        cat_ids = sorted(c["id"] for c in coco["categories"])
        category_mapping = {cid: i for i, cid in enumerate(cat_ids)}

    # Group annotations by image
    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    for img_id, anns in anns_by_image.items():
        img = id_to_image[img_id]
        stem = Path(img["file_name"]).stem
        lines = []
        for ann in anns:
            cls = category_mapping[ann["category_id"]]
            cx, cy, w, h = coco_bbox_to_yolo(
                ann["bbox"], img["width"], img["height"]
            )
            lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        with open(os.path.join(output_label_dir, f"{stem}.txt"), "w") as f:
            f.write("\n".join(lines))

    print(f"Converted {len(anns_by_image)} images to YOLO format in '{output_label_dir}'")


# Test with a tiny synthetic COCO file
sample_coco = {
    "images": [{"id": 1, "file_name": "test.jpg", "width": 640, "height": 480}],
    "categories": [{"id": 1, "name": "dog"}, {"id": 2, "name": "cat"}],
    "annotations": [
        {"id": 1, "image_id": 1, "category_id": 1, "bbox": [120, 80, 200, 150]},
        {"id": 2, "image_id": 1, "category_id": 2, "bbox": [400, 200, 100, 80]},
    ],
}

os.makedirs("/tmp/coco_test", exist_ok=True)
with open("/tmp/coco_test/annotations.json", "w") as f:
    json.dump(sample_coco, f)

convert_coco_to_yolo("/tmp/coco_test/annotations.json", "/tmp/yolo_labels")

with open("/tmp/yolo_labels/test.txt") as f:
    print("Output YOLO label file:")
    print(f.read())

## Dataset Structure for YOLO

The `ultralytics` trainer expects this folder layout:

```
dataset/
  images/
    train/   <- training images (.jpg, .png)
    val/     <- validation images
  labels/
    train/   <- one .txt per training image
    val/     <- one .txt per validation image
  dataset.yaml
```

The `dataset.yaml` tells the trainer where the data lives and what the class names are.
Images and label files must share the same filename stem (e.g., `cat001.jpg` and `cat001.txt`).
Images with no annotations still need an empty `.txt` file.

## Augmentation Pipelines for Detection

Standard image augmentation (flip, crop, color jitter) needs special handling for detection:
when you flip an image, every bounding box must flip too. When you crop, boxes outside
the crop boundary need to be dropped or clipped.

Albumentations handles this automatically when you declare augmentations with `BboxParams`.
You pass the bounding boxes in the same call as the image, and get transformed boxes back.

In [ ]:
# Detection augmentation pipeline with Albumentations
transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
        A.RandomScale(scale_limit=0.2, p=0.5),
        A.PadIfNeeded(min_height=640, min_width=640, p=1.0),
        A.RandomCrop(height=640, width=640, p=1.0),
    ],
    # bbox_params tells Albumentations boxes are in YOLO normalized format
    # and to drop any box whose area falls below min_area after the transform
    bbox_params=A.BboxParams(
        format="yolo",
        min_area=100,
        min_visibility=0.3,
        label_fields=["class_labels"],
    ),
)


def augment_sample(image_np, bboxes_yolo, class_labels):
    """
    image_np: HxWx3 numpy array
    bboxes_yolo: list of [cx, cy, w, h] normalized
    class_labels: list of int class IDs
    Returns augmented image, new bboxes, new labels.
    """
    result = transform(
        image=image_np,
        bboxes=bboxes_yolo,
        class_labels=class_labels,
    )
    return result["image"], result["bboxes"], result["class_labels"]


# Quick demo on a synthetic image
dummy_img = np.zeros((480, 640, 3), dtype=np.uint8)
dummy_img[100:200, 150:300] = [200, 100, 50]  # a colored rectangle

bboxes = [[0.35, 0.31, 0.23, 0.21]]  # [cx, cy, w, h] normalized
labels = [0]

aug_img, aug_boxes, aug_labels = augment_sample(dummy_img, bboxes, labels)
print(f"Input boxes:  {bboxes}")
print(f"Output boxes: {list(aug_boxes)}")
print(f"Output shape: {aug_img.shape}")

## Hands-On: Build a Synthetic Dataset and Fine-Tune

We'll generate a small dataset programmatically: 40 images with drawn rectangles
representing two classes (red boxes and blue boxes). Then we fine-tune YOLOv8n
for a few epochs and check the validation mAP.

In [ ]:
DATASET_DIR = Path("/tmp/synthetic_detection_dataset")
CLASSES = ["red_box", "blue_box"]
IMG_SIZE = 416
N_TRAIN = 32
N_VAL = 8


def generate_image(img_size=416, n_objects=2):
    """Generate a white image with colored rectangles and return image + YOLO labels."""
    img = Image.new("RGB", (img_size, img_size), color=(240, 240, 240))
    draw = ImageDraw.Draw(img)
    labels = []

    for _ in range(n_objects):
        cls = random.randint(0, 1)
        color = (220, 50, 50) if cls == 0 else (50, 80, 220)

        w = random.randint(40, 120)
        h = random.randint(40, 120)
        x1 = random.randint(0, img_size - w - 1)
        y1 = random.randint(0, img_size - h - 1)
        x2, y2 = x1 + w, y1 + h

        draw.rectangle([x1, y1, x2, y2], fill=color, outline=(0, 0, 0), width=2)

        cx = (x1 + x2) / 2 / img_size
        cy = (y1 + y2) / 2 / img_size
        nw = w / img_size
        nh = h / img_size
        labels.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

    return img, labels


def create_dataset_split(split_dir, n_images):
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for i in range(n_images):
        img, labels = generate_image(IMG_SIZE, n_objects=random.randint(1, 3))
        stem = f"img_{i:04d}"
        img.save(img_dir / f"{stem}.jpg")
        with open(lbl_dir / f"{stem}.txt", "w") as f:
            f.write("\n".join(labels))


# Create train and val splits
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

create_dataset_split(DATASET_DIR / "train", N_TRAIN)
create_dataset_split(DATASET_DIR / "val", N_VAL)
print(f"Dataset created: {N_TRAIN} train, {N_VAL} val images")

In [ ]:
dataset_yaml = f"""path: {DATASET_DIR}
train: train/images
val: val/images

nc: {len(CLASSES)}
names: {CLASSES}
"""

yaml_path = DATASET_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(dataset_yaml)

print("dataset.yaml:")
print(dataset_yaml)

In [ ]:
# Fine-tune YOLOv8n on the synthetic dataset
# YOLOv8n is the nano variant: fastest, lowest accuracy, but fine for this demo
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=5,
    imgsz=IMG_SIZE,
    batch=8,
    lr0=0.01,
    name="synthetic_finetune",
    project="/tmp/yolo_runs",
    verbose=False,
)

print("Training complete.")

### YOLO Fine-Tuning: All Key Arguments Explained

The call below shows every commonly used training argument with an inline comment. The synthetic dataset from the previous cell is used here, but the same call works on any YOLO-format dataset by changing `data`.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Start from pretrained COCO weights (transfer learning, not training from scratch)
finetune_model = YOLO('yolov8n.pt')

train_results = finetune_model.train(
    data=str(yaml_path),      # path to dataset.yaml (defined in the earlier cell)
    epochs=10,                # total training epochs
    imgsz=IMG_SIZE,           # input image size (square); model resizes internally
    batch=8,                  # images per batch (reduce if you hit OOM)
    lr0=0.01,                 # initial learning rate
    lrf=0.01,                 # final lr as a fraction of lr0 (cosine schedule)
    momentum=0.937,           # SGD momentum
    weight_decay=0.0005,      # L2 regularization
    warmup_epochs=3.0,        # linear LR warmup for the first N epochs
    patience=50,              # early stopping: stop if no improvement after N epochs
    save=True,                # save checkpoints (best.pt and last.pt)
    save_period=-1,           # save every N epochs (-1 = only best and last)
    project='/tmp/yolo_runs', # root directory for all run outputs
    name='finetune_explained',# subdirectory name for this run
    exist_ok=True,            # overwrite if the run directory already exists
    pretrained=True,          # use the pretrained COCO weights as initialization
    optimizer='SGD',          # optimizer: 'SGD', 'Adam', 'AdamW', 'auto'
    verbose=False,            # suppress per-batch logs (True to see them)
    seed=42,                  # random seed for reproducibility
    # --- Augmentation parameters (can also be set in the YAML) ---
    hsv_h=0.015,              # HSV hue augmentation range
    hsv_s=0.7,                # HSV saturation augmentation range
    hsv_v=0.4,                # HSV value augmentation range
    degrees=0.0,              # rotation range (degrees)
    translate=0.1,            # translation fraction
    scale=0.5,                # scale jitter range
    flipud=0.0,               # vertical flip probability
    fliplr=0.5,               # horizontal flip probability
    mosaic=1.0,               # mosaic augmentation probability (0=off, 1=always)
    mixup=0.0,                # mixup augmentation probability
    copy_paste=0.0,           # copy-paste augmentation probability
)

print('Fine-tuning complete.')
print(f'Best model saved to: {train_results.save_dir}/weights/best.pt')

## Data Augmentation Parameters in the YAML Config

Instead of passing augmentation parameters directly to `model.train()`, you can include them in a separate augmentation YAML file. This makes the configuration reproducible and shareable. The block below shows the standard mosaic, mixup, and HSV parameters with their typical values and what each one does.

In [ ]:
# Evaluate on the validation set
metrics = model.val(data=str(yaml_path), verbose=False)

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")

# After only 5 epochs on a tiny synthetic dataset,
# mAP50 should be high (>0.8) because the task is easy:
# solid-colored rectangles on a gray background.

## Common Pitfalls

**Learning rate too high.** YOLOv8 defaults are tuned for the COCO scale.
When fine-tuning on small datasets, `lr0=0.001` or lower often works better.

**Forgetting to check image/label pairing.** If an image has no matching `.txt` file,
it's silently skipped or treated as background-only. Run a sanity check to confirm
every image has a corresponding label.

**Class imbalance.** If one class has 10x more instances than another,
the model will optimize primarily for the majority class. Use the class weights
parameter or oversample the minority class.

**Wrong normalization of boxes after resize.** YOLO coordinates are already normalized,
so they don't change with image resize. But if you manually compute boxes from
pixel coordinates after an augmentation, make sure you divide by the *new* image size,
not the original.